# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library—all by referencing dataset entities using their `@id` fields.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

# Show the dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets, their `@id`s, fields, and `@id`s. Each table-like part of the dataset is a record set (referenced by `@id`).

In [ ]:
# List all record sets with their @ids

if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = getattr(metadata, 'recordSet', getattr(metadata, 'record_set', []))

# Preview all record sets with their @id and name
recordset_info = []
for rs in record_sets:
    if hasattr(rs, 'id'):
        rs_id = rs.id
    elif hasattr(rs, '@id'):
        rs_id = rs['@id']
    elif '@id' in rs:
        rs_id = rs['@id']
    else:
        rs_id = str(rs)
    name = getattr(rs, 'name', getattr(rs, 'label', rs_id))
    print(f"Record set @id: {rs_id}   name: {name}")
    recordset_info.append((rs_id, name))

if not recordset_info:
    print("No record sets found. Attempting to infer from dataset.records()...")
    # mlcroissant exposes all available record sets via dataset.available_record_sets
    available_record_sets = dataset.available_record_sets
    for rs_id in available_record_sets:
        print(f"Record set @id: {rs_id}")

### Inspect fields for a selected record set
Pick a record set `@id` from the list above and list its fields, columns, and their `@id`s. Replace `<record_set_id>` below with the actual ID you want to inspect.

In [ ]:
# If no record_sets detected above, list from available_record_sets

record_set_id = None

if recordset_info:
    record_set_id = recordset_info[0][0] # first available
else:
    all_rs = dataset.available_record_sets
    if len(all_rs) > 0:
        record_set_id = all_rs[0]

if record_set_id:
    print(f"Inspecting record set: {record_set_id}")
    # List a preview of records (from generator)
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i > 2:
            break
    # List available fields/columns
    fields = dataset.schema['@graph']
    print("\nFields/columns in this record set:")
    found_fields = []
    for entity in fields:
        if entity.get('@type') in ['cr:Field', 'cr:Column', 'Field', 'Column']:
            if 'cr:recordSet' in entity and entity['cr:recordSet'] == record_set_id:
                print(f"  @id: {entity['@id']}, name: {entity.get('schema:name', '')}")
                found_fields.append(entity['@id'])
    if not found_fields:
        # try a fallback: print keys from first record
        print('No explicit fields found, using first record keys:')
        for i, record in enumerate(dataset.records(record_set=record_set_id)):
            for k in record.keys():
                print(f'  field/column: {k}')
            break
else:
    print("No record set found to inspect.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame using record set and field `@id`s. We'll demonstrate with the primary tabular record set.

In [ ]:
# Build a list of all available record set @ids
record_set_ids = []
if recordset_info:
    record_set_ids = [item[0] for item in recordset_info]
elif hasattr(dataset, 'available_record_sets'):
    record_set_ids = dataset.available_record_sets

# Load all record sets into dataframes
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '{rs_id}' with {len(df)} rows and columns: {list(df.columns)}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Show the columns and preview the first record set's DataFrame
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print("\nFirst record set @id:", first_rs_id)
    print("Columns:", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No DataFrames were loaded.")

## 4. Exploratory Data Analysis (EDA)
We'll explore filtering numeric fields, normalizing values, and grouping. Be sure to reference fields by their `@id`s!

In [ ]:
import numpy as np

# Pick the primary record set as before
primary_rs_id = list(dataframes.keys())[0]
df = dataframes[primary_rs_id]

# Show numeric fields/column @ids
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns for filtering/EDA: {numeric_cols}")
if len(numeric_cols) == 0:
    # Fallback: attempt to convert any columns that look numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            pass
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Pick the first available numeric column as a demo
if numeric_cols:
    numeric_field_id = numeric_cols[0]
else:
    print("No numeric fields available for EDA.")
    numeric_field_id = None

# Filtering, normalization, grouping
if numeric_field_id:
    threshold = df[numeric_field_id].mean() # set threshold as mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalizing
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Find a likely categorical/grouping field
    group_field = None
    objcols = df.select_dtypes(include=['object']).columns.tolist()
    for col in objcols:
        if df[col].nunique() > 1 and df[col].nunique() < (len(df)//2):
            group_field = col
            break
    if group_field:
        grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field}':")
        print(grouped.head())
else:
    print("No numeric field found for EDA demonstration.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field and its breakdown by a grouping field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if group_field found
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found, so no visualization produced.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library, discovering its internal structure and records exclusively via schema `@id` references.

You can now build upon this notebook to perform domain-specific analyses on the provided colorectal cancer survivor dataset, referencing fields and record sets by their standardized Croissant `@id`s for maximum reproducibility and clarity.